<a href="https://colab.research.google.com/github/JorgeMiceli1967/IA-SCRAPPING/blob/main/Relevamiento_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Relevamiento web temático para estado del arte

# ============================================================
# RELEVAMIENTO WEB TEMÁTICO PARA ESTADO DEL ARTE
# Compatible con Google Colab y Python local
# ============================================================
#
# Requisitos:
#   pip install requests pandas openpyxl beautifulsoup4 lxml
#
# Qué hace:
# - busca resultados web por campo temático usando SerpAPI
# - visita cada URL y extrae texto básico
# - calcula puntaje de relevancia
# - clasifica tipo de actor (gobierno, academia, ONG, etc.)
# - detecta país aproximado por dominio
# - exporta resultados a CSV y Excel
# - genera tablas resumen
#
# IMPORTANTE:
# - Necesitás una API key de SerpAPI: https://serpapi.com/
# - No inventa métricas ni clasificaciones “inteligentes”: usa reglas explícitas
#
# AUTOR: Jorge Eduardo Miceli (jorgemiceli@hotmail.com) - VERSION 2
#
# CAMBIOS DE VERSION 2:
# Redefinición de campos temáticos: de 4 categorías a 7 (separación conceptual de servicios, academia, normativa y jurisprudencia).
# Cambio completo del esquema de salida: nuevas columnas normalizadas (CAMPO, ACTOR, PAÍS, etc.).
# Incorporación de la columna CONSULTA para registrar cada búsqueda.
# Detección automática de PAÍS a partir del dominio.
# Unificación de DESCRIPCIÓN PÁGINA (eliminación de duplicidad con meta description).
# Renombrado general de variables (formato consistente, en mayúsculas).
# Ajuste de ordenamiento, filtros y deduplicación a los nuevos nombres de columnas.
# Agregado de nuevos resúmenes: por país y por consulta.
# Construcción explícita de la fila final (estructura controlada del dataset).
# Mejora general de la estructura para análisis (más trazabilidad y segmentación temática).
# Filtrado de caracteres ilegales
# ============================================================

# !pip install -q requests pandas openpyxl beautifulsoup4 lxml

import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse

# Si corrés en Colab, esto permite descarga automática y acceso a secrets
try:
    from google.colab import files, userdata
    IN_COLAB = True
    API_KEY = userdata.get("SERPAPI_KEY")
except Exception:
    IN_COLAB = False
    API_KEY = None

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

if not API_KEY:
    raise ValueError("No se encontró la clave SERPAPI_KEY en los Secrets de Colab.")

print("Clave cargada correctamente.")

LANGUAGE = "es"
COUNTRIES = ["es", "ar", "mx"]   # Modificado para permitir múltiples países, por ejemplo: España, Argentina, México
RESULTS_PER_QUERY = 10
SLEEP_BETWEEN_SEARCHES = 1.2
SLEEP_BETWEEN_PAGES = 0.8
REQUEST_TIMEOUT = 25
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0 Safari/537.36"
)

OUTPUT_CSV = "relevamiento_estado_del_arte.csv"
OUTPUT_XLSX = "relevamiento_estado_del_arte.xlsx"
OUTPUT_SUMMARY_XLSX = "relevamiento_resumen.xlsx"

VISIT_PAGES = True
MAX_TEXT_LENGTH = 12000
MIN_RELEVANCE_SCORE = 2
DROP_DUPLICATES_BY_URL = True

# ============================================================
# CAMPOS TEMÁTICOS
# ============================================================

FIELDS = {
    "IA y Datos MF": {
        "keywords": [
            "inteligencia artificial ministerio fiscal",
            "AI prosecutor office digital evidence",
            "inteligencia artificial fiscalía prueba digital",
            "AI criminal justice prosecution service",
            "automatización procesos ministerio público",
            "analítica de datos justicia penal",
            "AI prosecution digital transformation",
            "procesamiento de prueba digital fiscalía",
        ],
        "relevance_terms": [
            "inteligencia artificial", "ai", "machine learning", "algoritmo",
            "analítica de datos", "data analytics", "prueba digital",
            "digital evidence", "fiscalía", "ministerio fiscal",
            "ministerio público", "prosecution", "criminal justice",
            "judicial analytics", "automatización", "proceso penal"
        ],
        "institution_terms": [
            "ministerio fiscal", "ministerio público", "fiscalía",
            "prosecutor", "prosecution service", "attorney general",
            "ministerio de justicia", "court", "tribunal"
        ]
    },

    "Servicios Jurídicos": {
        "keywords": [
            "legal tech España",
            "legal technology Spain AI law",
            "software jurídico inteligencia artificial España",
            "compliance AI software Spain",
            "document review legal AI Spain",
            "contract analysis legal AI Spain",
            "litigation analytics Spain legal tech",
            "despacho abogados inteligencia artificial España"
        ],
        "relevance_terms": [
            "legal tech", "legal ai", "legal analytics", "compliance",
            "document review", "contract analysis", "litigation analytics",
            "despacho", "firma", "software jurídico", "derecho digital",
            "automatización legal", "legal software", "servicios jurídicos"
        ],
        "institution_terms": [
            "despacho", "firma", "empresa", "consultora", "legal tech",
            "provider", "software", "platform", "startup"
        ]
    },

    "Oferta académica": {
        "keywords": [
            "máster inteligencia artificial derecho España",
            "posgrado derecho tecnología España",
            "grupo investigación derecho inteligencia artificial España",
            "revista inteligencia artificial derecho España",
            "master artificial intelligence law university",
            "law school artificial intelligence program",
            "research group AI law university",
            "facultad de derecho inteligencia artificial"
        ],
        "relevance_terms": [
            "máster", "master", "posgrado", "postgraduate",
            "grupo de investigación", "research group", "law school",
            "facultad de derecho", "universidad", "university",
            "revista", "journal", "programa", "curso", "seminario"
        ],
        "institution_terms": [
            "universidad", "university", "facultad", "faculty",
            "research center", "grupo de investigación", ".edu", ".ac."
        ]
    },

    "Herramientas IA": {
        "keywords": [
            "large language models legal applications",
            "herramientas de IA para derecho",
            "AI legal assistant tools",
            "legal AI software",
            "LLM legal research",
            "sistemas expertos jurídicos",
            "AI agent legal workflow",
            "self-hosted legal AI",
            "open source LLM legal",
        ],
        "relevance_terms": [
            "llm", "large language model", "generative ai", "ia generativa",
            "ai agent", "agente de ia", "sistema experto", "expert system",
            "nlp", "natural language processing", "legal ai", "chatbot",
            "open source", "self-hosted", "on premise", "local deployment",
            "herramienta", "tool", "software", "platform"
        ],
        "institution_terms": [
            "platform", "software", "tool", "herramienta", "modelo",
            "modelo fundacional", "startup", "empresa", "provider"
        ]
    },

    "Bibliografía": {
        "keywords": [
            "artificial intelligence criminal justice research",
            "inteligencia artificial proceso penal artículo",
            "AI judicial decision making paper",
            "machine learning criminal justice journal",
            "AI criminal law review article",
            "AI sentencing research",
            "predictive policing legal scholarship",
            "academic article artificial intelligence law"
        ],
        "relevance_terms": [
            "paper", "research article", "journal", "law review",
            "academic article", "artículo", "revista",
            "legal scholarship", "proceso penal", "criminal law",
            "bibliografía", "doctrina", "research", "scholarship"
        ],
        "institution_terms": [
            "journal", "law review", "revista", "repositorio",
            "repository", "ssrn", "heinonline", "scielo"
        ]
    },

    "Normativa": {
        "keywords": [
            "regulación inteligencia artificial justicia",
            "AI regulation judiciary",
            "normativa inteligencia artificial proceso penal",
            "guidelines AI courts",
            "boletín oficial inteligencia artificial justicia",
            "artificial intelligence legal framework courts",
            "judicial AI policy regulation",
            "AI governance tribunals"
        ],
        "relevance_terms": [
            "regulación", "normativa", "guidelines", "policy",
            "marco regulatorio", "legal framework", "gobernanza",
            "boletín oficial", "decreto", "resolución", "ley",
            "tribunal", "court", "judiciary", "justice"
        ],
        "institution_terms": [
            "boe", "gov", "gob", "official gazette", "ministerio",
            "tribunal", "court", "judiciary", "justice"
        ]
    },

    "Jurisprudencia": {
        "keywords": [
            "jurisprudencia inteligencia artificial proceso penal",
            "AI case law criminal justice",
            "sentencia inteligencia artificial tribunal",
            "court decision artificial intelligence law",
            "case law artificial intelligence criminal procedure",
            "jurisprudencia algoritmo justicia",
            "sentencia IA derecho",
            "tribunal inteligencia artificial jurisprudencia"
        ],
        "relevance_terms": [
            "jurisprudencia", "case law", "sentencia", "judgment",
            "tribunal", "court", "decision", "ruling",
            "proceso penal", "criminal procedure", "criminal justice",
            "inteligencia artificial", "algoritmo"
        ],
        "institution_terms": [
            "tribunal", "court", "judiciary", "justice", "supreme court",
            "case law", "sentencia", "jurisprudencia"
        ]
    }
}

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalize_space(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""


def detect_country_from_domain(domain: str) -> str:
    """
    Detección simple y heurística de país a partir del dominio.
    Prioriza ccTLD y algunos dominios institucionales frecuentes.
    """
    d = (domain or "").lower()

    country_map = {
        ".ar": "Argentina",
        ".es": "España",
        ".mx": "México",
        ".cl": "Chile",
        ".uy": "Uruguay",
        ".br": "Brasil",
        ".co": "Colombia",
        ".pe": "Perú",
        ".ec": "Ecuador",
        ".bo": "Bolivia",
        ".py": "Paraguay",
        ".ve": "Venezuela",
        ".cr": "Costa Rica",
        ".pa": "Panamá",
        ".gt": "Guatemala",
        ".hn": "Honduras",
        ".sv": "El Salvador",
        ".ni": "Nicaragua",
        ".do": "República Dominicana",
        ".cu": "Cuba",
        ".us": "Estados Unidos",
        ".uk": "Reino Unido",
        ".fr": "Francia",
        ".de": "Alemania",
        ".it": "Italia",
        ".pt": "Portugal",
        ".nl": "Países Bajos",
        ".be": "Bélgica",
        ".ch": "Suiza",
        ".at": "Austria",
        ".ie": "Irlanda",
        ".ca": "Canadá",
        ".au": "Australia",
        ".nz": "Nueva Zelanda",
        ".jp": "Japón",
        ".cn": "China",
        ".in": "India",
        ".za": "Sudáfrica"
    }

    for suffix, country in country_map.items():
        if d.endswith(suffix):
            return country

    if "boe.es" in d:
        return "España"
    if ".gob.ar" in d or ".gov.ar" in d:
        return "Argentina"
    if ".gob.mx" in d or ".gov.mx" in d:
        return "México"
    if ".gov.br" in d:
        return "Brasil"
    if ".gov.uk" in d:
        return "Reino Unido"
    if ".gov" in d or ".edu" in d:
        return "Estados Unidos"

    # Adjusted to consider the first country in the list if no specific country is found
    if COUNTRIES and COUNTRIES[0] == "es":
        return "España"
    if COUNTRIES and COUNTRIES[0] == "ar":
        return "Argentina"
    if COUNTRIES and COUNTRIES[0] == "us":
        return "Estados Unidos"

    return "No determinado"


def classify_actor_type(domain: str, text: str) -> str:
    """
    Clasificación heurística simple y transparente.
    """
    d = (domain or "").lower()
    t = (text or "").lower()

    if any(x in d for x in [".gov", ".gob", "fiscal", "judiciary", "justice", "poderjudicial", "boe.es"]):
        return "gobierno/justicia"
    if any(x in d for x in [".edu", ".ac.", "universidad", "university"]) or \
       any(x in t for x in ["universidad", "university", "facultad", "faculty", "research group", "grupo de investigación"]):
        return "academia"
    if any(x in d for x in [".org", ".int"]) or \
       any(x in t for x in ["ngo", "ong", "nonprofit", "asociación", "fundación", "foundation"]):
        return "ong/organización"
    if any(x in t for x in ["software", "platform", "plataforma", "startup", "company", "empresa", "provider"]):
        return "empresa/proveedor"
    return "otro"


def count_term_hits(text: str, terms: list[str]) -> int:
    text_lower = (text or "").lower()
    hits = 0
    for term in terms:
        if term.lower() in text_lower:
            hits += 1
    return hits


def score_result(title: str, snippet: str, page_text: str, relevance_terms: list[str], institution_terms: list[str]) -> dict:
    """
    Puntaje simple y explícito.
    """
    title_hits = count_term_hits(title, relevance_terms)
    snippet_hits = count_term_hits(snippet, relevance_terms)
    page_hits = count_term_hits(page_text, relevance_terms)
    institution_hits = count_term_hits(page_text, institution_terms) + count_term_hits(snippet, institution_terms)

    score = (title_hits * 3) + (snippet_hits * 2) + page_hits + institution_hits

    return {
        "hits_titulo": title_hits,
        "hits_snippet": snippet_hits,
        "hits_texto": page_hits,
        "hits_institucionales": institution_hits,
        "puntaje_relevancia": score
    }


def search_serpapi(query: str, api_key: str, num_results: int = 10, country_code: str = "es") -> list[dict]:
    endpoint = "https://serpapi.com/search.json"
    params = {
        "engine": "google",
        "q": query,
        "api_key": api_key,
        "hl": LANGUAGE,
        "gl": country_code, # Use the passed country_code
        "num": num_results,
    }
    response = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    data = response.json()
    return data.get("organic_results", [])


def fetch_page_text(url: str) -> dict:
    """
    Extrae título, meta description y texto visible básico de una página.
    Si falla, devuelve vacío.
    """
    headers = {"User-Agent": USER_AGENT}
    try:
        r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        content_type = r.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return {"page_title": "", "meta_description": "", "page_text": "", "http_status": r.status_code}

        soup = BeautifulSoup(r.text, "lxml")

        for tag in soup(["script", "style", "noscript", "svg", "img", "header", "footer", "nav", "aside"]):
            tag.extract()

        page_title = normalize_space(soup.title.get_text()) if soup.title else ""

        meta_description = ""
        md = soup.find("meta", attrs={"name": "description"})
        if md and md.get("content"):
            meta_description = normalize_space(md["content"])

        text = normalize_space(soup.get_text(" "))
        if len(text) > MAX_TEXT_LENGTH:
            text = text[:MAX_TEXT_LENGTH]

        return {
            "page_title": page_title,
            "meta_description": meta_description,
            "page_text": text,
            "http_status": r.status_code
        }
    except Exception:
        return {"page_title": "", "meta_description": "", "page_text": "", "http_status": ""}


def normalize_search_result(field_name: str, query: str, item: dict) -> dict:
    url = item.get("link", "").strip()
    title = normalize_space(item.get("title", ""))
    snippet = normalize_space(item.get("snippet", ""))
    domain = extract_domain(url)

    return {
        "campo": field_name,
        "query": query,
        "titulo_busqueda": title,
        "snippet_busqueda": snippet,
        "url": url,
        "dominio": domain
    }

# Helper function to clean text for Excel export
def clean_excel_text(text):
    if isinstance(text, str):
        # Remove illegal XML characters which openpyxl doesn't like
        # See: https://www.w3.org/TR/REC-xml/#charsets
        # Excludes: #x0-#x8, #xb, #xc, #xe-#x1f
        # This regex matches characters that are NOT allowed in XML 1.0
        illegal_chars_re = re.compile(u'[\x00-\x08\x0b\x0c\x0e-\x1f]')
        text = illegal_chars_re.sub('', text)
    return text

# ============================================================
# RECOLECCIÓN
# ============================================================

if API_KEY == "PON_AQUI_TU_API_KEY":
    raise ValueError("Reemplazá API_KEY por tu clave real de SerpAPI.")

rows = []

for field_name, field_conf in FIELDS.items():
    keywords = field_conf["keywords"]
    relevance_terms = field_conf["relevance_terms"]
    institution_terms = field_conf["institution_terms"]

    print(f"\n=== Campo: {field_name} ===")

    for country_code in COUNTRIES: # Iterate through each country
        print(f"  Buscando para el país: {country_code.upper()}")
        for query in keywords:
            print(f"    Buscando: {query}")
            try:
                results = search_serpapi(query, API_KEY, RESULTS_PER_QUERY, country_code=country_code) # Pass country_code to search_serpapi

                for item in results:
                    row = normalize_search_result(field_name, query, item)
                    if not row["url"]:
                        continue

                    page_data = {"page_title": "", "meta_description": "", "page_text": "", "http_status": ""}
                    if VISIT_PAGES:
                        page_data = fetch_page_text(row["url"])
                        time.sleep(SLEEP_BETWEEN_PAGES)

                    combined_text = " ".join([
                        row["titulo_busqueda"],
                        row["snippet_busqueda"],
                        page_data["page_title"],
                        page_data["meta_description"],
                        page_data["page_text"]
                    ])

                    score_data = score_result(
                        title=row["titulo_busqueda"],
                        snippet=row["snippet_busqueda"],
                        page_text=page_data["page_text"],
                        relevance_terms=relevance_terms,
                        institution_terms=institution_terms
                    )

                    actor_type = classify_actor_type(row["dominio"], combined_text)
                    country = detect_country_from_domain(row["dominio"]) # This will still prioritize TLDs, then fall back to the first in COUNTRIES

                    final_row = {
                        "CAMPO": field_name, # Use field_name directly for clarity
                        "CONSULTA": row["query"],
                        "PAÍS BÚSQUEDA": country_code.upper(), # Add the country code used for the search
                        "ACTOR": actor_type,
                        "PAÍS DETECTADO": country, # Keep the detected country from domain
                        "URL": row["url"],
                        "DOMINIO": row["dominio"],
                        "TÍTULO": row["titulo_busqueda"],
                        "TÍTULO PÁGINA": page_data["page_title"],
                        "RESÚMEN BÚSQUEDA": row["snippet_busqueda"],
                        "DESCRIPCIÓN PÁGINA": page_data["meta_description"],
                        "TEXTO PÁGINA": page_data["page_text"],
                        "HITS TITULO": score_data["hits_titulo"],
                        "HITS SNIPPET": score_data["hits_snippet"],
                        "HITS TEXTO": score_data["hits_texto"],
                        "HITS INSTITUCIONALES": score_data["hits_institucionales"],
                        "PUNTAJE": score_data["puntaje_relevancia"]
                    }
                    rows.append(final_row)

            except requests.HTTPError as e:
                print(f"Error HTTP con '{query}' para país {country_code}: {e}")
            except requests.RequestException as e:
                print(f"Error de conexión con '{query}' para país {country_code}: {e}")
            except Exception as e:
                print(f"Error inesperado con '{query}' para país {country_code}: {e}")

            time.sleep(SLEEP_BETWEEN_SEARCHES)

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

if df.empty:
    print("No se recuperaron resultados.")
else:
    if DROP_DUPLICATES_BY_URL:
        df = df.sort_values(by="PUNTAJE", ascending=False)
        df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)

    df = df[df["PUNTAJE"] >= MIN_RELEVANCE_SCORE].reset_index(drop=True)

    df = df.sort_values(
        by=["CAMPO", "PUNTAJE", "ACTOR"],
        ascending=[True, False, True]
    ).reset_index(drop=True)

    preferred_cols = [
        "CAMPO",
        "CONSULTA",
        "PAÍS BÚSQUEDA", # Added new column
        "ACTOR",
        "PAÍS DETECTADO", # Changed column name for clarity
        "URL",
        "DOMINIO",
        "TÍTULO",
        "TÍTULO PÁGINA",
        "RESÚMEN BÚSQUEDA",
        "DESCRIPCIÓN PÁGINA",
        "TEXTO PÁGINA",
        "HITS TITULO",
        "HITS SNIPPET",
        "HITS TEXTO",
        "HITS INSTITUCIONALES",
        "PUNTAJE"
    ]
    df = df[[c for c in preferred_cols if c in df.columns]]

    # Apply cleaning to relevant text columns before export
    text_columns_to_clean = [
        "TÍTULO", "TÍTULO PÁGINA", "RESÚMEN BÚSQUEDA",
        "DESCRIPCIÓN PÁGINA", "TEXTO PÁGINA"
    ]
    for col in text_columns_to_clean:
        if col in df.columns:
            df[col] = df[col].apply(clean_excel_text)

    print("\nVista previa de resultados:")
    print(df.head(15).to_string(index=False))

    # ========================================================
    # TABLAS RESUMEN
    # ========================================================

    summary_by_field = (
        df.groupby("CAMPO")
          .agg(
              resultados=("URL", "count"),
              dominios_unicos=("DOMINIO", "nunique"),
              puntaje_promedio=("PUNTAJE", "mean")
          )
          .reset_index()
    )

    # Update summary tables to include 'PAÍS BÚSQUEDA' for a more granular view
    summary_by_actor = (
        df.groupby(["CAMPO", "PAÍS BÚSQUEDA", "ACTOR"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["CAMPO", "PAÍS BÚSQUEDA", "cantidad"], ascending=[True, True, False])
    )

    top_domains = (
        df.groupby(["CAMPO", "PAÍS BÚSQUEDA", "DOMINIO"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["CAMPO", "PAÍS BÚSQUEDA", "cantidad"], ascending=[True, True, False])
    )

    summary_by_country = (
        df.groupby(["CAMPO", "PAÍS BÚSQUEDA", "PAÍS DETECTADO"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["CAMPO", "PAÍS BÚSQUEDA", "cantidad"], ascending=[True, True, False])
    )

    summary_by_query = (
        df.groupby(["CAMPO", "PAÍS BÚSQUEDA", "CONSULTA"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["CAMPO", "PAÍS BÚSQUEDA", "cantidad"], ascending=[True, True, False])
    )

    print("\nResumen por campo:")
    print(summary_by_field.to_string(index=False))

    print("\nResumen por campo, país de búsqueda y tipo de actor:")
    print(summary_by_actor.head(30).to_string(index=False))

    print("\nResumen por campo, país de búsqueda y país detectado:")
    print(summary_by_country.head(30).to_string(index=False))

    print("\nResumen por campo, país de búsqueda y consulta:")
    print(summary_by_query.head(30).to_string(index=False))

    # ========================================================
    # EXPORTACIÓN
    # ========================================================

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="resultados", index=False)

    with pd.ExcelWriter(OUTPUT_SUMMARY_XLSX, engine="openpyxl") as writer:
        summary_by_field.to_excel(writer, sheet_name="resumen_campo", index=False)
        summary_by_actor.to_excel(writer, sheet_name="resumen_actor", index=False)
        top_domains.to_excel(writer, sheet_name="top_dominios", index=False)
        summary_by_country.to_excel(writer, sheet_name="resumen_pais", index=False)
        summary_by_query.to_excel(writer, sheet_name="resumen_consulta", index=False)

    print(f"\nArchivos guardados:")
    print(f"- {OUTPUT_CSV}")
    print(f"- {OUTPUT_XLSX}")
    print(f"- {OUTPUT_SUMMARY_XLSX}")

    if IN_COLAB:
        files.download(OUTPUT_CSV)
        files.download(OUTPUT_XLSX)
        files.download(OUTPUT_SUMMARY_XLSX)


Clave cargada correctamente.

=== Campo: IA y Datos MF ===
Buscando: inteligencia artificial ministerio fiscal
Buscando: AI prosecutor office digital evidence
Buscando: inteligencia artificial fiscalía prueba digital
Buscando: AI criminal justice prosecution service
Buscando: automatización procesos ministerio público
Buscando: analítica de datos justicia penal
Buscando: AI prosecution digital transformation
Buscando: procesamiento de prueba digital fiscalía

=== Campo: Servicios Jurídicos ===
Buscando: legal tech España
Buscando: legal technology Spain AI law
Buscando: software jurídico inteligencia artificial España
Buscando: compliance AI software Spain
Buscando: document review legal AI Spain
Buscando: contract analysis legal AI Spain
Buscando: litigation analytics Spain legal tech
Buscando: despacho abogados inteligencia artificial España

=== Campo: Oferta académica ===
Buscando: máster inteligencia artificial derecho España
Buscando: posgrado derecho tecnología España
Buscando: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>